# Week 4 — DeepSpeed & ZeRO Mathematics

By the end of Week 3 we know how to split a model across devices. This week we ask a sharper question: **how little memory per device can we get away with while keeping the optimizer step semantically identical to vanilla Adam?**

The answer is the **Zero Redundancy Optimizer (ZeRO)** family.

## Learning objectives

1. Derive the **16N-byte** baseline memory footprint of mixed-precision Adam.
2. Quantify the per-stage savings of ZeRO-1, ZeRO-2, and ZeRO-3.
3. Model the CPU/NVMe offload bandwidth budget.
4. Configure DeepSpeed via JSON and verify the savings empirically.


## 1. The 16N Baseline

For a model with $N$ parameters trained with mixed-precision Adam:

| Quantity                       | Bytes per parameter |
|--------------------------------|---------------------|
| FP16 parameter                 | 2                   |
| FP16 gradient                  | 2                   |
| FP32 master parameter          | 4                   |
| FP32 Adam first moment $m$     | 4                   |
| FP32 Adam second moment $v$    | 4                   |
| **Total state**                | **16 bytes**        |

So $M_{\text{state}} = 16 N$ bytes — **before any activations**.

For a 1.3B-parameter model this is **20.8 GiB**, leaving very little budget on an A100-80GB for activations, especially at long sequence length.


## 2. ZeRO-1: Partition the Optimizer State

Stage 1 partitions only the **fp32 master + Adam $m$ + Adam $v$** (12 bytes per parameter) across $N_{\text{GPU}}$ ranks. Each rank still holds the full fp16 parameter and fp16 gradient.

$$
M_{\text{ZeRO-1}} \;=\; 2N + 2N + \frac{12 N}{N_{\text{GPU}}}
\;=\; 4N + \frac{12N}{N_{\text{GPU}}}.
$$

For $N_{\text{GPU}} = 8$ and $N = 1.3 \times 10^9$ this drops state from 20.8 GiB to **7.1 GiB** — a $2.9 \times$ saving.

The optimizer step now runs only over the local 1/$N_{\text{GPU}}$-th slice; an AllGather broadcasts the updated parameters back.


## 3. ZeRO-2: Also Partition Gradients

Stage 2 additionally partitions the 2N bytes of fp16 gradients, requiring a **reduce-scatter** instead of an all-reduce during the backward pass:

$$
M_{\text{ZeRO-2}} \;=\; 2N + \frac{2N + 12N}{N_{\text{GPU}}}
\;=\; 2N + \frac{14N}{N_{\text{GPU}}}.
$$

For the same 1.3B / 8-GPU setting: **4.4 GiB** — a $4.7 \times$ saving over baseline.


## 4. ZeRO-3: Also Partition the Parameters

Stage 3 partitions everything, including the fp16 parameters. During the forward pass, each layer's parameters must be **all-gathered** just-in-time and released immediately after the matmul (and again during backward).

$$
M_{\text{ZeRO-3}} \;=\; \frac{16N}{N_{\text{GPU}}} + M_{\text{activations}}.
$$

For 1.3B / 8 GPUs: **2.6 GiB** — a $8 \times$ saving. The trade-off is **extra communication**: each parameter is moved $\sim 2 \times \text{num\_layers}$ times per step instead of once.


In [ ]:
# %% Memory accounting calculator
def zero_memory_per_gpu(N: int, n_gpu: int, stage: int) -> dict:
    if stage == 0:
        param = 2 * N
        grad = 2 * N
        opt = 12 * N
    elif stage == 1:
        param = 2 * N
        grad = 2 * N
        opt = 12 * N / n_gpu
    elif stage == 2:
        param = 2 * N
        grad = 2 * N / n_gpu
        opt = 12 * N / n_gpu
    elif stage == 3:
        param = 2 * N / n_gpu
        grad = 2 * N / n_gpu
        opt = 12 * N / n_gpu
    else:
        raise ValueError(stage)
    total = param + grad + opt
    return {'param_GiB': param / 2**30, 'grad_GiB': grad / 2**30,
            'opt_GiB': opt / 2**30, 'total_GiB': total / 2**30}

for stage in range(4):
    out = zero_memory_per_gpu(N=1_300_000_000, n_gpu=8, stage=stage)
    print(f"Stage {stage}: {out}")


## 5. CPU / NVMe Offload

ZeRO-Infinity (ZeRO-3 + Offload) moves the partitioned state to host memory or NVMe SSD. The bandwidth budget is:

* **GPU↔CPU (PCIe Gen5 x16):** ~64 GB/s per direction.
* **GPU↔NVMe (PCIe Gen4 x4):** ~7 GB/s.

For a forward+backward step that takes $t_{\text{step}}$ seconds, offload is feasible iff

$$
\text{offloaded\_bytes\_per\_step} \;\lesssim\; t_{\text{step}} \cdot B_{\text{offload}}.
$$

In practice, CPU offload sustains $\sim 90\%$ of in-GPU throughput on H100; NVMe offload sustains $\sim 40\text{–}60\%$.


In [ ]:
// configs/deepspeed/zero3_offload.json (illustrative)
{
  "fp16": { "enabled": true, "loss_scale_window": 1000 },
  "zero_optimization": {
    "stage": 3,
    "offload_optimizer": { "device": "cpu", "pin_memory": true },
    "offload_param":     { "device": "cpu", "pin_memory": true },
    "overlap_comm": true,
    "contiguous_gradients": true,
    "reduce_bucket_size": 5e8,
    "stage3_prefetch_bucket_size": 5e8,
    "stage3_param_persistence_threshold": 1e6
  },
  "gradient_clipping": 1.0,
  "wall_clock_breakdown": true
}


## 6. Activation Checkpointing — The Other Memory Lever

Independent of ZeRO, activations can be traded for compute via **gradient checkpointing**: keep only every $\sqrt{L}$-th activation and recompute the rest on the backward pass. Memory drops from $O(L)$ to $O(\sqrt{L})$ at a compute cost of $\sim 33\%$ extra FLOPs.

Combining ZeRO-3 + activation checkpointing + FP8 is the standard recipe for training $\geq 70$B models on 8×H100.


## 7. Exercises

1. Plot the memory-per-GPU as a function of stage $\in \{0, 1, 2, 3\}$ for $N \in \{350M, 1.3B, 7B\}$ on 8 GPUs.
2. Run a small model with `zero_stage=0` and `zero_stage=3` and compare wallclock + memory. By what fraction does throughput change?
3. Derive the optimal activation-checkpoint interval given $t_{\text{recompute}}$ and a memory budget $M_{\text{budget}}$.
